In [61]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler , LabelEncoder , OneHotEncoder
from sklearn.model_selection import train_test_split

In [62]:
df = pd.read_csv('student_digital_life.csv')
df.head()

,student_id,age,gender,study_hours_per_day,smartphone_usage_hours,social_media_hours,gaming_hours,streaming_hours,sleep_hours,exercise_hours,class_attendance_percent,assignment_completion_percent,caffeine_intake_cups,mental_health_status,parent_education_level,internet_quality,motivation_level,final_exam_score
0,1,21,Female,3.01,0.26,1.77,0.26,1.71,5.32,0.70,64.87,74.11,2,Good,Masters,Average,6.32,82.70
1,2,23,Female,5.84,8.02,3.30,0.96,0.00,5.99,1.99,85.38,80.68,3,Average,Masters,Poor,2.52,85.65
2,3,20,Female,7.80,10.13,0.00,2.23,2.54,6.36,1.82,76.15,79.69,2,Good,HighSchool,Poor,3.98,88.14
3,4,20,Female,0.00,1.15,1.32,4.19,0.27,7.86,0.18,84.41,79.07,0,Average,HighSchool,Average,4.77,54.81
4,5,24,Male,7.23,1.39,2.21,4.67,2.75,7.88,0.28,81.13,65.40,4,Good,HighSchool,Average,8.77,84.34


fit()  →  Sadece istatistikleri öğrenir, veriyi DEĞİŞTİRMEZ
Kullanım amacı: Scaler'ı yalnızca TRAIN seti üzerinde eğitmek.Test setinin istatistiklerini modele sızdırmamak (data leakage önleme)

In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 18 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   student_id                     15000 non-null  int64  
 1   age                            15000 non-null  int64  
 2   gender                         15000 non-null  object 
 3   study_hours_per_day            15000 non-null  float64
 4   smartphone_usage_hours         15000 non-null  float64
 5   social_media_hours             15000 non-null  float64
 6   gaming_hours                   15000 non-null  float64
 7   streaming_hours                15000 non-null  float64
 8   sleep_hours                    15000 non-null  float64
 9   exercise_hours                 15000 non-null  float64
 10  class_attendance_percent       15000 non-null  float64
 11  assignment_completion_percent  15000 non-null  float64
 12  caffeine_intake_cups           15000 non-null 

In [64]:
numeric_cols = ['study_hours_per_day', 'smartphone_usage_hours',
                'sleep_hours', 'class_attendance_percent',
                'assignment_completion_percent', 'final_exam_score']

In [65]:
X = df[numeric_cols]
X

,study_hours_per_day,smartphone_usage_hours,sleep_hours,class_attendance_percent,assignment_completion_percent,final_exam_score
0,3.01,0.26,5.32,64.87,74.11,82.70
1,5.84,8.02,5.99,85.38,80.68,85.65
2,7.80,10.13,6.36,76.15,79.69,88.14
3,0.00,1.15,7.86,84.41,79.07,54.81
4,7.23,1.39,7.88,81.13,65.40,84.34
...,...,...,...,...,...,...
14995,4.40,5.47,8.26,88.61,73.31,66.39
14996,6.21,11.89,7.07,80.00,73.04,73.38
14997,6.61,5.82,3.91,80.68,90.72,98.00
14998,3.67,4.58,8.35,81.76,48.75,68.07


In [66]:
X_train , X_test = train_test_split(
    X , 
    test_size=0.2,
    random_state=42
)
scaler = StandardScaler()
scaler.fit(X_train)

StandardScaler()

In [67]:
print("=== fit() sonucu öğrenilen istatistikler ===")
for col, mean, std in zip(numeric_cols, scaler.mean_, scaler.scale_ ):
    print(f"  {col:<38} mean={mean:.3f}  std={std:.3f}")

=== fit() sonucu öğrenilen istatistikler ===
  study_hours_per_day                    mean=4.513  std=1.980
  smartphone_usage_hours                 mean=5.505  std=2.438
  sleep_hours                            mean=6.995  std=1.460
  class_attendance_percent               mean=84.673  std=9.450
  assignment_completion_percent          mean=79.399  std=13.922
  final_exam_score                       mean=80.363  std=18.371


In [68]:
# Test verisine fit() yaparsak data leakage gerçekleşir.
X_trained_scaled = scaler.transform(X_train)
X_tested_scaled = scaler.transform(X_test)

print("\n=== transform() sonucu (ilk 3 satır) ===")
print("Train (scaled):\n", X_trained_scaled[:3].round(3))
print("Test  (scaled):\n", X_tested_scaled[:3].round(3))



=== transform() sonucu (ilk 3 satır) ===
Train (scaled):
 [[ 0.312 -1.262 -0.702  0.06  -1.359 -0.819]
 [ 0.433  0.416 -0.12  -0.063 -0.625  1.069]
 [ 1.554  0.359 -0.524 -0.563 -0.684  1.069]]
Test  (scaled):
 [[-1.057 -0.88   0.352 -1.179 -0.478 -0.155]
 [ 1.559 -0.72   0.195  1.622 -1.048  1.069]
 [ 0.711 -0.183 -0.038 -0.948  0.769  0.51 ]]


In [69]:
# fit_transform() sadece train seti için kullanılır.
scaler2 = StandardScaler()
X_trained_scaled2 = scaler.fit_transform(X_train)
X_tested_scaled2 = scaler.transform(X_test)

In [70]:
print("\n=== fit_transform() sonucu (ilk 3 satır) ===")
print(X_trained_scaled2[:3].round(3))

# Doğrulama: her iki yöntem aynı sonucu verir
assert np.allclose(X_trained_scaled, X_trained_scaled2), "Sonuçlar farklı!"
print("\n✔ fit().transform()  ==  fit_transform()  → Sonuçlar aynı!")


=== fit_transform() sonucu (ilk 3 satır) ===
[[ 0.312 -1.262 -0.702  0.06  -1.359 -0.819]
 [ 0.433  0.416 -0.12  -0.063 -0.625  1.069]
 [ 1.554  0.359 -0.524 -0.563 -0.684  1.069]]

✔ fit().transform()  ==  fit_transform()  → Sonuçlar aynı!


Kategorik Sütunlara LabelEncoder Örneği

In [74]:
le = LabelEncoder()

# fit()         → unique sınıfları öğren
le.fit(df['mental_health_status'])
print("\n=== LabelEncoder fit() → öğrenilen sınıflar ===")
print(le.classes_)

# transform()   → uygula
encoded = le.transform(df['mental_health_status'])
print("İlk 10 encode değer:", encoded[:10])

# fit_transform() → kısayol
encoded2 = LabelEncoder().fit_transform(df['mental_health_status'])
assert np.array_equal(encoded, encoded2)
print("✔ LabelEncoder fit_transform da aynı sonucu verdi!")


=== LabelEncoder fit() → öğrenilen sınıflar ===
['Average' 'Good' 'Poor']
İlk 10 encode değer: [1 0 1 0 1 1 1 0 1 0]
✔ LabelEncoder fit_transform da aynı sonucu verdi!


In [72]:
type(df[['mental_health_status']])

pandas.core.frame.DataFrame